# Apache Iceberg com Apache Spark (PySpark)

Demonstração de **DDL, INSERT, UPDATE e DELETE** em uma tabela Iceberg.

## Cenário

Tabela única **`carros`**: o cadastro de veículos de uma seguradora. O dataset (`data/carros.csv`, com 8 registros) foi criado pelo grupo a partir do exemplo visto em aula.

### Modelo ER

```mermaid
erDiagram
    CARROS {
        string placa PK
        string modelo
        string chassi
        string marca
        int ano
        string cor
    }
```

| Coluna | Tipo | Descrição |
|---|---|---|
| placa | STRING | Placa do veículo (chave) |
| modelo | STRING | Modelo |
| chassi | STRING | Número do chassi |
| marca | STRING | Fabricante |
| ano | INT | Ano de fabricação |
| cor | STRING | Cor |

## 1. SparkSession com Apache Iceberg

O JAR do Iceberg é baixado do Maven na primeira execução (`spark.jars.packages`). Usamos um **catálogo local do tipo `hadoop`** chamado `local`, que grava tudo na pasta `warehouse/iceberg`.

In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession

WAREHOUSE = (Path("..") / "warehouse" / "iceberg").resolve()

spark = (
    SparkSession.builder.appName("apache-iceberg")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.2")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", str(WAREHOUSE))
    .getOrCreate()
)

# Recria a tabela a cada execução
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.seguradora")
spark.sql("DROP TABLE IF EXISTS local.seguradora.carros PURGE")

spark.version

## Dados de origem (CSV)

In [ ]:
df = spark.read.csv(
    "../data/raw/carros.csv",
    header=True,
    schema="placa STRING, modelo STRING, chassi STRING, marca STRING, ano INT, cor STRING",
)
df.createOrReplaceTempView("carros_csv")
df.show()

## 2. DDL: criação da tabela Iceberg

In [ ]:
spark.sql("""
    CREATE TABLE local.seguradora.carros (
        placa  STRING,
        modelo STRING,
        chassi STRING,
        marca  STRING,
        ano    INT,
        cor    STRING
    )
    USING iceberg
""")

spark.sql("INSERT INTO local.seguradora.carros SELECT * FROM carros_csv")  # carga inicial
spark.sql("SELECT * FROM local.seguradora.carros ORDER BY placa").show()

## 3. INSERT

Inclusão de um carro novo.

In [ ]:
spark.sql("""
    INSERT INTO local.seguradora.carros
    VALUES ('KLM2468', 'ONIX', '11223344556', 'GM', 2023, 'PRATA')
""")
spark.sql("SELECT * FROM local.seguradora.carros ORDER BY placa").show()

## 4. UPDATE

O ECO SPORT (placa `EEE1056`) foi pintado de vermelho.

In [ ]:
spark.sql("UPDATE local.seguradora.carros SET cor = 'VERMELHO' WHERE placa = 'EEE1056'")
spark.sql("SELECT * FROM local.seguradora.carros WHERE placa = 'EEE1056'").show()

## 5. DELETE

Exclusão dos carros fabricados antes de 2015 (CLIO 2011 e PUNTO 2013).

In [ ]:
spark.sql("DELETE FROM local.seguradora.carros WHERE ano < 2015")
spark.sql("SELECT * FROM local.seguradora.carros ORDER BY placa").show()

## 6. Snapshots e time travel

Cada operação gera um novo **snapshot**, consultável pela tabela de metadados `.snapshots`.

In [ ]:
snapshots = spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM local.seguradora.carros.snapshots
    ORDER BY committed_at
""")
snapshots.show(truncate=False)

Consultando a tabela como ela estava **logo após a carga inicial** (primeiro snapshot):

In [ ]:
primeiro_snapshot = snapshots.first()["snapshot_id"]
spark.sql(
    f"SELECT * FROM local.seguradora.carros VERSION AS OF {primeiro_snapshot} ORDER BY placa"
).show()

## 7. Arquivos gerados

Dados em **Parquet** (`data/`) + metadados em **JSON/Avro** (`metadata/`).

In [ ]:
tabela = WAREHOUSE / "seguradora" / "carros"
for arquivo in sorted(tabela.rglob("*")):
    if arquivo.is_file() and not arquivo.name.endswith(".crc"):
        print(arquivo.relative_to(tabela))

In [ ]:
spark.stop()